# GreenBond GDELT Sentiment — Europe (Germany & Denmark)

This notebook constructs the daily climate sentiment series for **Germany** and **Denmark** used in the European sovereign green bond analysis.

It replicates the Asia GDELT pipeline (India & Indonesia) with country-specific adaptations.

| Parameter | Value |
|---|---|
| Countries | Germany, Denmark |
| Period | January 2019 – May 2026 |
| Source | GDELT 2.0 Global Knowledge Graph (GKG) via BigQuery |
| Primary variable | GDELT V2Tone (normalised to [−1, 1]) |
| Treatment events | Germany green Bund: 2 Sep 2020; Denmark green bond: 21 Jan 2022; EuGB agreement: 28 Feb 2023 |
| Output | `daily_sentiment_gdelt_europe.csv` → input to `GreenBond_Regression_Europe.ipynb` |


---
## Cell 1 — Install Dependencies

Installs required packages for BigQuery access, NLP scoring, and data handling.
Only needs to run once per Colab session.


In [1]:
!pip install -q transformers torch pandas requests tqdm google-cloud-bigquery db-dtypes

---
## Cell 2 — Imports and Configuration

Sets up file paths, study parameters, and key treatment event dates.

**Treatment events (used as event markers in the sentiment timeline):**
- **2 September 2020** — Germany inaugurates its first sovereign green bond (10Y green Bund, €6.5bn, twin-bond design). This is the primary treatment event for the German panel.
- **21 January 2022** — Denmark inaugurates its first sovereign green bond (0% Nov 2031, twin-bond design, DKK 10bn). Primary treatment event for the Danish panel.
- **28 February 2023** — European Parliament and Council reach provisional agreement on the EU Green Bond Standard (EuGB), the first legally binding green bond framework tied to the EU Taxonomy. Secondary event affecting both markets.

**Output directories:**
- `data/processed/gdelt_europe/` — sentiment CSVs
- `output/gdelt_europe/` — visualisation PNGs


In [ ]:
import os
import pandas as pd
import torch
from tqdm.notebook import tqdm

# ── Mount Google Drive ─────────────────────────────────────────────────
from google.colab import drive
drive.mount("/content/drive")

# ── Thesis root on Drive ──────────────────────────────────────────────
ROOT = "/content/drive/MyDrive/Thesis"

# ── Output directories ────────────────────────────────────────────────
# Separate from Asia pipeline to avoid overwriting existing files
DATA_DIR   = os.path.join(ROOT, "data", "processed", "gdelt")          # matches regression notebook path
OUTPUT_DIR = os.path.join(ROOT, "output", "gdelt_europe")
os.makedirs(DATA_DIR,   exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Study parameters ──────────────────────────────────────────────────
PROJECT_ID = "project-c7cff37e-dfac-4b4b-bc2"   # Google Cloud project
COUNTRIES  = ["Germany", "Denmark"]

# Start from 2019 to give 18 months of pre-treatment data
# before Germany's inaugural green Bund (Sep 2020)
START_DATE = 20190101000000
END_DATE   = 20260501235959

# ── Key treatment events ──────────────────────────────────────────────
# Used as vertical markers in the sentiment timeline chart
EVENTS = {
    "2020-09-02": "Germany\nGreen Bund",    # inaugural 10Y twin-bond
    "2022-01-21": "Denmark\nGreen Bond",    # inaugural 0% Nov 2031 twin-bond
    "2023-02-28": "EuGB\nAgreement",        # EU Green Bond Standard provisional deal
}

print(f"Data directory   : {DATA_DIR}")
print(f"Output directory : {OUTPUT_DIR}")
print(f"Countries        : {', '.join(COUNTRIES)}")
print(f"Period           : {START_DATE} → {END_DATE}")
print(f"Key events       : {list(EVENTS.keys())}")


---
## Cell 3 — GDELT Data Collection via Google BigQuery

Queries the GDELT 2.0 Global Knowledge Graph (GKG) public table on BigQuery.

**Why BigQuery?**  
The GDELT DOC API was attempted first but abandoned due to rate limiting (HTTP 429 errors) and unreliable bulk collection. BigQuery provides direct access to the full GDELT dataset with no rate limits.

**Date format note:**  
GDELT stores dates as 14-digit integers (`YYYYMMDDHHMMSS`). Standard `YYYYMMDD` format returns 0 results.

**Country filter:**  
Uses GDELT geocoding identifiers for precision over plain text matching:
- `Germany#GM` — Federal Republic of Germany
- `Denmark#DA` — Kingdom of Denmark

Note: Germany generates substantially more climate-related news coverage than Denmark, reflecting both country size and Germany's role as a major green bond issuer and EU climate policy leader.

**Theme filter:**  
Applied directly in SQL to avoid pulling millions of irrelevant rows. Covers CLIMATE, ENV_, ENERGY, and GREEN themes — same filter as the Asia notebook for comparability.

**Expected volume:** ~85,000–100,000 articles per country (significantly more than India/Indonesia due to European media coverage density).


In [ ]:
from google.colab import auth
auth.authenticate_user()

from google.cloud import bigquery
client = bigquery.Client(project=PROJECT_ID)

query = """
SELECT
  DATE,
  SourceCommonName  AS source,
  DocumentIdentifier AS url,
  V2Tone            AS tone,
  Locations         AS locations,
  Themes            AS themes
FROM `gdelt-bq.gdeltv2.gkg`
WHERE DATE BETWEEN 20190101000000 AND 20260501235959
  AND (
    Locations LIKE '%Germany#GM%'
    OR Locations LIKE '%Denmark#DA%'
  )
  AND (
    Themes LIKE '%CLIMATE%'
    OR Themes LIKE '%ENV_%'
    OR Themes LIKE '%ENERGY%'
    OR Themes LIKE '%GREEN%'
  )
"""

print("Querying GDELT GKG (Germany & Denmark, 2019–2026)...")
print("Estimated runtime: 3–5 minutes\n")

df = client.query(query).to_dataframe()
print(f"Records retrieved : {len(df):,}")
print(f"DATE min          : {df['DATE'].min()}")
print(f"DATE max          : {df['DATE'].max()}")

# Assign country label based on GDELT location codes
df["country"] = df["locations"].apply(
    lambda x: "Germany" if "Germany#GM" in str(x) else "Denmark"
)
print("\nArticles by country:")
print(df["country"].value_counts())

# Save raw output
raw_path = f"{DATA_DIR}/gdelt_raw_articles_europe.csv"
df.to_csv(raw_path, index=False)
print(f"\nSaved → {raw_path}")


---
## Cell 4 — Sample Articles for Efficient ClimateBERT Scoring

The raw GDELT pull for Germany and Denmark contains far more articles than the Asia dataset — Germany alone generates several hundred articles per day on climate topics. Scoring all of them with ClimateBERT would take many hours on GPU.

**Sampling strategy:** 1,000 articles per country per month, sampled randomly with a fixed seed (42) for reproducibility. This yields approximately 86 months × 2 countries × 1,000 = ~172,000 articles — statistically representative while remaining computationally feasible.

**Note on Germany vs Denmark coverage:**  
Germany will have far more raw articles per month than Denmark. The 1,000-article cap ensures both countries contribute equally to the monthly aggregate, preventing Germany from dominating the panel.


In [ ]:
raw_df = pd.read_csv(f"{DATA_DIR}/gdelt_raw_articles_europe.csv")

# Parse date from 14-digit GDELT timestamp
raw_df["date"] = pd.to_datetime(
    raw_df["DATE"].astype(str).str[:8],
    format="%Y%m%d",
    errors="coerce"
)
raw_df["month"] = raw_df["date"].dt.to_period("M")
raw_df = raw_df.dropna(subset=["date"])

print(f"Before sampling : {len(raw_df):,} articles")
print(f"Date range      : {raw_df['date'].min().date()} → {raw_df['date'].max().date()}")

# Sample 1,000 articles per country per month
sampled_df = (
    raw_df.groupby(["country", "month"])
    .apply(lambda x: x.sample(min(len(x), 1000), random_state=42))
    .reset_index(drop=True)
)

print(f"After sampling  : {len(sampled_df):,} articles")
print(f"Months covered  : {sampled_df['month'].nunique()}")
print("\nSampled articles by country:")
print(sampled_df["country"].value_counts())

sampled_df.to_csv(f"{DATA_DIR}/gdelt_raw_articles_europe.csv", index=False)
print(f"\nSaved sampled articles → {DATA_DIR}/gdelt_raw_articles_europe.csv")


---
## Cell 5 — Load ClimateBERT

Loads the `climatebert/distilroberta-base-climate-sentiment` model from HuggingFace.

**Why ClimateBERT?**  
Unlike general-purpose sentiment models (e.g. VADER, TextBlob), ClimateBERT is fine-tuned on climate-related financial disclosures and policy documents. It distinguishes between vague environmental claims ("we are committed to sustainability") and credible policy commitments ("binding 2030 emissions target legislated") — a critical distinction for measuring political credibility signals.

**Output labels:**
- `positive` → +1.0 (ambitious, credible climate commitment)
- `neutral` → 0.0 (factual/ambiguous)
- `negative` → −1.0 (regressive or hostile climate signal)

**Important caveat (from Asia analysis):**  
ClimateBERT applied to GDELT `themes` columns (structured topic codes rather than natural language) returned predominantly neutral scores in the Asia chapter. The same limitation applies here. ClimateBERT scores are therefore retained only as a robustness check column — **the primary regression variable is GDELT V2Tone** (`sentiment_gdelt`).


In [ ]:
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification

MODEL_NAME = "climatebert/distilroberta-base-climate-sentiment"

print(f"Loading {MODEL_NAME}...")
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)
model      = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
device     = 0 if torch.cuda.is_available() else -1

classifier = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
    device=device,
    truncation=True,
    max_length=128,
)

print(f"Device : {'GPU ✓' if device == 0 else 'CPU (slower)'}")
print("ClimateBERT ready.")

---
## Cell 6 — Score Articles with ClimateBERT

Scores each article's `themes` field using ClimateBERT in batches of 64.

**Runtime:** ~15–20 minutes on T4 GPU for ~172,000 articles.

**Note on scoring the `themes` column:**  
GDELT `themes` contains structured topic codes (e.g. `CLIMATE_CHANGE;RENEWABLE_ENERGY;ENV_NUCLEAR`) rather than natural language sentences. ClimateBERT was designed for prose text, so scores on GDELT theme codes are treated as supplementary only. The GDELT `V2Tone` score (computed in Cell 7) is the primary continuous sentiment variable, validated for bulk news analysis across all article types.

**You can skip this cell** if you already have `gdelt_climatebert_scored_europe.csv` on Drive,
or if you want to use V2Tone only (recommended given the theme-code limitation above).
In that case, proceed directly to Cell 7 and load from `gdelt_raw_articles_europe.csv`.


In [ ]:
LABEL_MAP  = {"positive": 1.0, "neutral": 0.0, "negative": -1.0}
BATCH_SIZE = 64

raw_df = pd.read_csv(f"{DATA_DIR}/gdelt_raw_articles_europe.csv")
print(f"Loaded {len(raw_df):,} articles for scoring")

texts  = raw_df["themes"].fillna("").tolist()
scores = []

print(f"Scoring {len(texts):,} texts in batches of {BATCH_SIZE}...")
for i in tqdm(range(0, len(texts), BATCH_SIZE)):
    batch = [str(t)[:512] for t in texts[i:i+BATCH_SIZE]]
    try:
        results = classifier(batch)
        scores.extend([LABEL_MAP.get(r["label"].lower(), 0.0) for r in results])
    except Exception as e:
        print(f"  Batch {i//BATCH_SIZE} error: {e}")
        scores.extend([0.0] * len(batch))

raw_df["climatebert_score"] = scores
scored_path = f"{DATA_DIR}/gdelt_climatebert_scored_europe.csv"
raw_df.to_csv(scored_path, index=False)
print(f"\nSaved scored articles → {scored_path}")
print("\nClimateBERT score distribution:")
print(raw_df["climatebert_score"].value_counts())


---
## Cell 7 — Build Daily Sentiment Series

Aggregates article-level V2Tone scores to a daily country-level sentiment series.

This produces the key independent variable **S\_{c,t}** used in the European regression:

$$\\text{Sentiment}_{c,t} = \\frac{1}{N_{c,t}} \\sum_{i=1}^{N_{c,t}} \\frac{\\text{V2Tone}_{i,c,t}}{10} \\in [-1, 1]$$

**Two sentiment measures are constructed:**
- `sentiment_gdelt` — mean V2Tone score across all articles for country $c$ on day $t$, normalised to [−1, 1] by dividing by 10. **Primary regression variable.**
- `sentiment_climatebert` — mean ClimateBERT score. Retained as robustness check only.

**Signaling Noise proxy:**  
`sentiment_std` = daily standard deviation of V2Tone across articles. Days with high variance indicate contradictory political signals — some sources reporting positively, others negatively on the same climate policy. This is the H2 variable in the regression.

**For Germany:** expect ~30–50 articles/day with reasonable std.  
**For Denmark:** expect ~10–20 articles/day; std will be noisier due to lower volume.


In [ ]:
raw_df = pd.read_csv(f"{DATA_DIR}/gdelt_climatebert_scored_europe.csv")

raw_df["date"] = pd.to_datetime(
    raw_df["DATE"].astype(str).str[:8],
    format="%Y%m%d",
    errors="coerce"
)
raw_df = raw_df.dropna(subset=["date"])

raw_df["sentiment_gdelt"] = raw_df["tone"].apply(
    lambda x: max(-1.0, min(1.0, float(str(x).split(",")[0]) / 10.0))
    if pd.notna(x) else 0.0
)

print(f"Date range : {raw_df['date'].min().date()} → {raw_df['date'].max().date()}")
print(f"Total rows : {len(raw_df):,}")

daily_df = (
    raw_df.groupby(["country", "date"])
    .agg(
        sentiment_gdelt       = ("sentiment_gdelt",      "mean"),
        sentiment_climatebert = ("climatebert_score",    "mean"),
        sentiment_std         = ("sentiment_gdelt",      "std"),
        article_count         = ("sentiment_gdelt",      "count"),
        pct_positive          = ("sentiment_gdelt",      lambda x: (x > 0).mean()),
        pct_negative          = ("sentiment_gdelt",      lambda x: (x < 0).mean()),
    )
    .reset_index()
    .sort_values(["country", "date"])
    .reset_index(drop=True)
)

daily_path = f"{DATA_DIR}/daily_sentiment_gdelt_europe.csv"
daily_df.to_csv(daily_path, index=False)
print(f"\nSaved daily sentiment → {daily_path}")
print(f"Total country-days : {len(daily_df):,}")
print("\nSummary by country:")
print(daily_df.groupby("country").agg(
    days           = ("date",           "nunique"),
    mean_sentiment = ("sentiment_gdelt", "mean"),
    mean_noise     = ("sentiment_std",   "mean"),
    total_articles = ("article_count",   "sum"),
).round(4).to_string())


---
## Cell 8 — Sentiment Timeline Visualisation

Plots the daily GDELT V2Tone sentiment series for Germany and Denmark with a 30-day moving average and the three key treatment event markers:

- **2 September 2020** — Germany inaugurates its first sovereign green bond (10Y twin-bond, 0% coupon, €6.5bn). This marks the beginning of Europe's most systematic sovereign green bond programme.
- **21 January 2022** — Denmark inaugurates its first sovereign green bond (0% Nov 2031 twin-bond). Denmark becomes the second country after Germany to adopt the strict twin-bond format.
- **28 February 2023** — Provisional agreement on the EU Green Bond Standard. First legally binding green bond framework in Europe, tied to the EU Taxonomy. Affects both Germany and Denmark simultaneously.

A visual break or trend shift around any of these dates would be consistent with the hypothesis that political events affect market sentiment. The absence of such breaks (as found in Asia) would support the market maturity interpretation.


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

daily_df = pd.read_csv(f"{DATA_DIR}/daily_sentiment_gdelt_europe.csv")
daily_df["date"] = pd.to_datetime(daily_df["date"])

print(f"Date range : {daily_df['date'].min().date()} → {daily_df['date'].max().date()}")
print(daily_df.groupby("country")["date"].nunique().rename("days"))

COLORS = {"Germany": "#378ADD", "Denmark": "#E85D24"}

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
fig.suptitle(
    "Daily Climate Sentiment (GDELT V2Tone) — Germany & Denmark 2019–2026",
    fontsize=13, y=1.01
)

for ax, (country, color) in zip(axes, COLORS.items()):
    data = daily_df[daily_df["country"] == country].copy().sort_values("date")
    data["smooth"] = data["sentiment_gdelt"].rolling(30, min_periods=5).mean()

    ax.fill_between(data["date"], data["sentiment_gdelt"], alpha=0.15, color=color)
    ax.plot(data["date"], data["smooth"], color=color, linewidth=1.8, label="30-day MA")
    ax.axhline(0, color="gray", linewidth=0.6, linestyle="--")

    for date_str, label in EVENTS.items():
        ed = pd.Timestamp(date_str)
        ax.axvline(ed, color="black", linewidth=1.0, linestyle=":")
        ax.text(ed, -0.15, label, fontsize=7, ha="center", va="top",
                bbox=dict(boxstyle="round,pad=0.2", fc="white",
                          alpha=0.85, ec="gray", linewidth=0.5))

    ax.set_ylabel(country, fontsize=10, fontweight="bold", color=color)
    ax.set_ylim(-1.1, 1.1)
    ax.yaxis.set_major_locator(plt.MultipleLocator(0.5))
    ax.grid(axis="y", linewidth=0.4, alpha=0.5)
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(loc="upper right", fontsize=8)

axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
axes[-1].xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.xticks(rotation=45)
plt.xlabel("Date")
plt.tight_layout()

chart_path = f"{OUTPUT_DIR}/sentiment_timeline_europe.png"
plt.savefig(chart_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {chart_path}")


---
## Cell 9 — Download Output Files

Downloads all output files from Google Drive to your local machine.

**Files for regression notebook (`GreenBond_Regression_Europe.ipynb`):**
- `daily_sentiment_gdelt_europe.csv` — the daily S\_{c,t} series for Germany and Denmark, ready for regression. This is the only file the regression notebook needs from this pipeline.

**Supporting files:**
- `gdelt_raw_articles_europe.csv` — raw GDELT articles (sampled, 1,000/country/month)
- `gdelt_climatebert_scored_europe.csv` — articles with ClimateBERT robustness scores
- `sentiment_timeline_europe.png` — visualisation for thesis Figure (Chapter 3 or appendix)

**File location in repo:**  
`Thesis/data/processed/gdelt/daily_sentiment_gdelt_europe.csv`

**Note:** The regression notebook (`GreenBond_Regression_Europe.ipynb`) reads from
`data/processed/gdelt/` (not `gdelt_europe/`) — ensure the file is saved to the correct folder.


In [ ]:
from google.colab import files

output_files = [
    f"{DATA_DIR}/gdelt_raw_articles_europe.csv",
    f"{DATA_DIR}/gdelt_climatebert_scored_europe.csv",
    f"{DATA_DIR}/daily_sentiment_gdelt_europe.csv",
    f"{OUTPUT_DIR}/sentiment_timeline_europe.png",
]

for f_path in output_files:
    if os.path.exists(f_path):
        print(f"✓ Downloading: {os.path.basename(f_path)}")
        files.download(f_path)
    else:
        print(f"✗ Not found: {f_path}")

print("\n── Summary ──────────────────────────────────────────")
print("Key regression input : daily_sentiment_gdelt_europe.csv")
print("Save to repo at      : Thesis/data/processed/gdelt/")
print("Used by              : GreenBond_Regression_Europe.ipynb")
